# Suite1 Project5 Classification

**Dataset:** California Housing (binned)

---
Run each cell in order. All outputs are generated from real data.

In [ ]:
# Setup: imports and configuration
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

# Plotting style
sns.set_theme(style="whitegrid", palette="Set2")
plt.rcParams['figure.dpi'] = 120

# Helper: load dataset
def load_data(name):
    return pd.read_csv(f'https://raw.githubusercontent.com/ulink-ibcs/A4_ML_Projects/main/datasets/{name}')

print("✅ Setup complete!")

In [ ]:
"""Project 5: Classification Demo — Binned House Value"""
import sys, os; sys.path.insert(0, os.path.dirname(__file__))

df = load_csv('housing.csv')
OUTPUT_DIR = os.path.join(BASE_DIR, 'suite1_project5_outputs'); os.makedirs(OUTPUT_DIR, exist_ok=True)

## PROJECT 5: CLASSIFICATION DEMO — Predicting High/Low House Value

In [ ]:
# Convert regression to binary classification
median_val = df['MedHouseVal'].median()
df['PriceCategory'] = (df['MedHouseVal'] > median_val).astype(int)
print(f"Median house value: ${median_val*100:.0f}K")
print(f"Class distribution:")
print(f"  0 (Below median): {(df['PriceCategory']==0).sum():,} ({(df['PriceCategory']==0).mean():.1%})")
print(f"  1 (Above median): {(df['PriceCategory']==1).sum():,} ({(df['PriceCategory']==1).mean():.1%})")

features = ['MedInc', 'HouseAge', 'AveRooms', 'AveBedrms', 'Population', 'AveOccup', 'Latitude', 'Longitude']
X = df[features]
y = df['PriceCategory']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# ── 1. K-Nearest Neighbours ──

## 1. K-Nearest Neighbours (K-NN)

In [ ]:
results_knn = []
for k in [1, 3, 5, 7, 11, 15, 21, 31, 51, 101]:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_train_scaled, y_train)
    y_pred = knn.predict(X_test_scaled)
    acc = accuracy_score(y_test, y_pred)
    results_knn.append({'k': k, 'Accuracy': round(acc, 4)})
    print(f"  k={k:3d}: Accuracy = {acc:.4f}")

knn_df = pd.DataFrame(results_knn)
print(f"\nBest k: {knn_df.loc[knn_df['Accuracy'].idxmax(), 'k']} (acc={knn_df['Accuracy'].max():.4f})")

# Visualization: k vs accuracy
fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(knn_df['k'], knn_df['Accuracy'], 'o-', color='#2e86de', linewidth=2)
ax.axvline(x=knn_df.loc[knn_df['Accuracy'].idxmax(), 'k'], color='green', linestyle=':', alpha=0.7)
ax.set_xlabel('k (number of neighbours)', fontsize=12)
ax.set_ylabel('Test Accuracy', fontsize=12)
ax.set_title('K-NN: Effect of k on Classification Accuracy', fontsize=13)
ax.grid(True, alpha=0.3)
plt.tight_layout()
# 'p5_knn_accuracy')
plt.show()
print("[Saved] p5_knn_accuracy.png")

# Best K-NN model
best_knn = KNeighborsClassifier(n_neighbors=int(knn_df.loc[knn_df['Accuracy'].idxmax(), 'k']))
best_knn.fit(X_train_scaled, y_train)
y_pred_knn = best_knn.predict(X_test_scaled)

print(f"\n=== K-NN (k={best_knn.n_neighbors}) Confusion Matrix ===")
cm = confusion_matrix(y_test, y_pred_knn)
print(pd.DataFrame(cm, index=['Actual Low', 'Actual High'], columns=['Pred Low', 'Pred High']).to_string())
print(f"\n{classification_report(y_test, y_pred_knn, target_names=['Low Price', 'High Price'])}")

# ── 2. Decision Tree ──

## 2. Decision Tree Classifier

In [ ]:
for max_depth in [3, 5, 10, None]:
    dt = DecisionTreeClassifier(max_depth=max_depth, random_state=42)
    dt.fit(X_train_scaled, y_train)
    y_pred = dt.predict(X_test_scaled)
    train_acc = accuracy_score(y_train, dt.predict(X_train_scaled))
    test_acc = accuracy_score(y_test, y_pred)
    print(f"  max_depth={max_depth!s:5s}: Train Acc={train_acc:.4f}, Test Acc={test_acc:.4f}")

# Best tree (max_depth=5 for interpretability)
best_dt = DecisionTreeClassifier(max_depth=5, random_state=42)
best_dt.fit(X_train, y_train)
y_pred_dt = best_dt.predict(X_test)

print(f"\n=== Decision Tree (max_depth=5) Confusion Matrix ===")
cm_dt = confusion_matrix(y_test, y_pred_dt)
print(pd.DataFrame(cm_dt, index=['Actual Low', 'Actual High'], columns=['Pred Low', 'Pred High']).to_string())
print(f"\n{classification_report(y_test, y_pred_dt, target_names=['Low Price', 'High Price'])}")

# Feature importance
print(f"\n=== Feature Importance (Decision Tree) ===")
fi = pd.DataFrame({'Feature': features, 'Importance': best_dt.feature_importances_}).sort_values('Importance', ascending=False)
print(fi.to_string(index=False))

fig, ax = plt.subplots(figsize=(10, 5))
ax.barh(fi['Feature'], fi['Importance'], color='#2e86de', alpha=0.7)
ax.set_xlabel('Importance', fontsize=12)
ax.set_title('Decision Tree Feature Importance', fontsize=13)
ax.invert_yaxis()
ax.grid(True, alpha=0.3, axis='x')
plt.tight_layout()
# 'p5_feature_importance')
plt.show()
print("[Saved] p5_feature_importance.png")

# ── 3. Compare Models ──

## 3. Model Comparison

In [ ]:
comparison = pd.DataFrame({
    'Model': ['K-NN', 'Decision Tree'],
    'Accuracy': [accuracy_score(y_test, y_pred_knn), accuracy_score(y_test, y_pred_dt)],
    'Precision (High)': [
        __import__('sklearn.metrics', fromlist=['precision_score']).precision_score(y_test, y_pred_knn),
        __import__('sklearn.metrics', fromlist=['precision_score']).precision_score(y_test, y_pred_dt)],
    'Recall (High)': [
        __import__('sklearn.metrics', fromlist=['recall_score']).recall_score(y_test, y_pred_knn),
        __import__('sklearn.metrics', fromlist=['recall_score']).recall_score(y_test, y_pred_dt)],
}).round(4)
print(comparison.to_string(index=False))

results_final = {
    'knn_results': results_knn,
    'best_k': int(knn_df.loc[knn_df['Accuracy'].idxmax(), 'k']),
    'knn_accuracy': float(accuracy_score(y_test, y_pred_knn)),
    'dt_accuracy': float(accuracy_score(y_test, y_pred_dt)),
    'feature_importance': fi.to_dict('records'),
    'comparison': comparison.to_dict('records'),
}
    json.dump(results_final, f, indent=2, default=str)

print("Done.")